In [1]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

c:\Users\Ansh\OneDrive\Desktop\LANGRAPH\meraenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
load_dotenv()
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
load_dotenv()
llm=ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv('GROQ_API_KEY'),
    temperature=0
)

In [11]:
from typing_extensions import Annotated

from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
conn = sqlite3.connect(database = "chatbot.db",check_same_thread=False)
checkpoint = SqliteSaver(conn=conn)

class chatbot(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

def chat(state: chatbot):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

graph = StateGraph(chatbot)
graph.add_node('chat',chat)

graph.add_edge(START,'chat')
graph.add_edge('chat',END)

workflow = graph.compile(checkpointer = checkpoint)
print(workflow)




In [ ]:
thread_id = 1

while True:
    user_message = input("write your message: ")
    print(user_message)

    if user_message in ['exit','quit','stop']:
        break
    else:
        config = {'configurable':{'thread_id':thread_id}}
        response = workflow.invoke({'messages':[HumanMessage(content = user_message)]}, config=config)
        print("AI Message = ",response['messages'][-1].content)
        
        print(response)# in this all past msgs are stored in messages variabvle inside state

hi what is my ane 
AI Message =  You said "hi" again, and then asked about your name. We already established that your name is Ansh.
{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='c60a0bed-7117-4d00-9551-f85133708314'), AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.009469737, 'completion_tokens_details': None, 'prompt_time': 0.00205129, 'prompt_tokens_details': None, 'queue_time': 0.005676715, 'total_time': 0.011521027}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_9ca2574dca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b71a6-e630-7950-9ffe-b61bcfda24a9-0', usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44}), HumanMessage(content='hi my name ias ansh and 22 years old', additional_kwargs={